In [4]:
import psycopg2
from psycopg2 import sql

conn = psycopg2.connect(dbname='wikidata_change_classification', user='postgres', password='postgres', host='172.16.64.23', port='5432')

queries = [
    ("Number of files", 
     "SELECT COUNT(distinct file_path) FROM revision"),

    ("Number of changes", 
     "SELECT COUNT(*) FROM value_change"),
    
    ("Number of creates", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'CREATE'"),

    ("Number of deletes", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'DELETE'"),

    ("Number of updates", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'UPDATE'"),
    
    ("Number of value updates", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'UPDATE' AND change_target = ''"),
    
    ("Number of string updates", 
     """SELECT COUNT(*) FROM value_change 
        WHERE action = 'UPDATE' AND change_target = '' 
        AND datatype IN ('monolingualtext', 'string', 'external-id', 'url', 'commonsMedia', 
                         'geo-shape', 'tabular-data', 'math', 'musical-notation', 'unknown-values')"""),
    
    ("Number of entity updates", 
     """SELECT COUNT(*) FROM value_change 
        WHERE action = 'UPDATE' AND change_target = '' 
        AND datatype IN ('wikibase-item', 'wikibase-entityid', 'wikibase-property', 
                         'wikibase-lexeme', 'wikibase-sense', 'wikibase-form', 'entity-schema')"""),

    ("Number of quantity updates", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'UPDATE' AND change_target = '' AND datatype = 'quantity'"),
    
    ("Number of time updates", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'UPDATE' AND change_target = '' AND datatype = 'time'"),
    
    ("Number of globecoordinate updates", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'UPDATE' AND change_target = '' AND datatype = 'globecoordinate'"),

    ("Number of rank updates", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'UPDATE' AND change_target = 'rank'"),

    ("Number of rank creations", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'CREATE' AND change_target = 'rank'"),

    ("Number of rank deletions", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'DELETE' AND change_target = 'rank'"),
    
    ("Number of dt metadata updates", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'UPDATE' AND change_target != 'rank' AND change_target != ''"),

    ("Number of dt metadata creations", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'CREATE' AND change_target != 'rank' AND change_target != ''"),

    ("Number of dt metadata deletions", 
     "SELECT COUNT(*) FROM value_change WHERE action = 'DELETE' AND change_target != 'rank' AND change_target != ''"),
    
]

print("="*70)
print("DATABASE STATISTICS")
print("="*70)

try:
    cur = conn.cursor()
    
    for description, query in queries:
        cur.execute(query)
        result = cur.fetchone()[0]
        print(f"{description:.<50} {result:>15,}")
    
    cur.close()
    
except Exception as e:
    print(f"Error: {e}")
    
finally:
    conn.close()

print("="*70)


DATABASE STATISTICS
Number of files...................................             107
Number of changes.................................      59,641,183
Number of creates.................................      47,790,058
Number of deletes.................................       7,911,307
Number of updates.................................       3,939,818
Number of value updates...........................       2,760,666
Number of string updates..........................       1,550,206
Number of entity updates..........................         956,018
Number of quantity updates........................         121,727
Number of time updates............................         105,357
Number of globecoordinate updates.................          27,025
Number of rank updates............................         558,610
Number of rank creations..........................      21,151,877
Number of rank deletions..........................       3,457,076
Number of dt metadata updates.............

### TODO

- Esperar a que termina la clasificación de value_refinement / unrefinement
- Traer de vuelta datos para clustering
- Hacer clustering de vuelta
- Revisar ejemplos de clusters -> tratar de ponerlos en clases que ya tengo definidas o en nuevas

In [4]:
import pandas as pd

df = pd.read_csv('clustered_data/cluster_examples_entity.csv')
df['change_target'] = df['change_target'].astype(str).fillna('')
df['revision_id'] = df['revision_id'].astype(str)
df['property_id'] = df['property_id'].astype(int)
df['value_id'] = df['value_id'].astype(str)



In [ ]:
df[['cluster_id', 'property_id', 'property_label', 'old_value_label', 'new_value_label', 'old_value', 'new_value']].sort_values(by='cluster_id')

# Cluster 5: Value refinement / unrefinement / Rewording
# Cluster 6: Link fix (the same name but changes in formatting - e.g. hyphens/dashes)


,cluster_id,property_id,property_label,old_value_label,new_value_label,old_value,new_value
0,0,279,NaN,National Historic Site,National Register of Historic Places listed place,Q1258086,Q19558910
14,0,166,award received,NaN,"Order of Suvorov, 1st class",Q4375473,Q21292820
13,0,166,award received,NaN,honorary doctorate of the University of Salamanca,Q58041808,Q50614979
12,0,39,position held,NaN,Roman Catholic Bishop of Liege,Q110368311,Q23595624
11,0,166,award received,Grammy Awards,Grammy Award for Best Spoken Word Album for Ch...,Q41254,Q3113382
...,...,...,...,...,...,...,...
91,6,413,NaN,fullback,full-back,Q5508224,Q90173132
90,6,413,NaN,fullback,full-back,Q5508224,Q90173132
103,6,361,part of,States-General,States General,Q393676,Q1371388
96,6,413,NaN,fullback,full-back,Q5508224,Q90173132


In [33]:

df_ass = pd.read_csv('clustered_data/cluster_assignments_entity.csv', index_col=0)
df_ass['change_target'] = df_ass['change_target'].astype(str).fillna('').replace('nan', '')
df_ass['revision_id'] = df_ass['revision_id'].astype(str)
df_ass['property_id'] = df_ass['property_id'].astype(int)
df_ass['value_id'] = df_ass['value_id'].astype(str)

df_data = pd.read_parquet('data/changes_for_clustering_entity_update.parquet')
df_data['change_target'] = df_data['change_target'].astype(str).fillna('')
df_data['revision_id'] = df_data['revision_id'].astype(str)
df_data['property_id'] = df_data['property_id'].astype(int)
df_data['value_id'] = df_data['value_id'].astype(str)

df_data.head(10)

df = pd.merge(left=df_data, right=df_ass, how='left', on=['revision_id', 'property_id', 'value_id', 'change_target'])

In [34]:
df.head(10)

,revision_id,entity_id,entity_label,property_id,value_id,property_label,old_value,old_value_label,new_value,new_value_label,...,new_hash,timestamp,user_type,username,user_id,comment,num_changes_in_revision,entity_age_days,change_magnitude,cluster_id
0,5790806,7742,Louis XIV of France,26,q7742$14FD06E8-E4FD-4090-AC98-D983E9145412,spouse,Q131706,Maria Theresa of Austria,Q152549,Maria Theresa of Spain,...,0efc6712e0453549a0a4bf5227095256858bb565,2013-02-04 20:11:05+00:00,human,Tpt,4098,,1,0E-24,10000.0,5
1,5790851,7742,Louis XIV of France,21,q7742$CF391E8A-EF79-44BE-ABF1-268EBF64D68E,sex or gender,Q1174782,Masculine,Q1076509,masculinity,...,c28bb4a28613cc441dc4358bf926d4dde45fc8fd,2013-02-04 20:14:15+00:00,human,Tpt,4098,,1,0.002199074074074074070000,10000.0,1
2,5790976,1001,Mahatma Gandhi,27,q1001$9D56D3E7-3EB3-4515-A54F-621A5A3A1775,None,Q1091034,Indian Motorcycle,Q862086,Indians,...,f663ef4384ab4d8ddd9261c2148a80b6425de0cc,2013-02-04 20:24:22+00:00,human,Nizil Shah,9101,,1,0E-24,10000.0,1
3,5792062,23559,Benito Mussolini,7,q23559$2A285E63-827B-4C69-8624-643731B6FC05,None,Q640292,Vittorio Mussolini,Q549097,Arnaldo Mussolini,...,ab38e3fcf0ccd1a21553036c71b3e9bf65b85ec7,2013-02-04 21:15:18+00:00,human,Vennor,53525,,1,0E-24,10000.0,5
4,5792339,1001,Mahatma Gandhi,27,q1001$9D56D3E7-3EB3-4515-A54F-621A5A3A1775,None,Q862086,Indians,Q668,India,...,6516b8e3c28dd3c8081231d1cd3164427a63d386,2013-02-04 21:26:51+00:00,human,Zolo,4610,,1,0.043391203703703703700000,10000.0,1
5,5792362,63733,Karl Adam,27,q63733$D027F7B9-B5AD-44F6-A690-FA354B2F2D76,None,Q42884,Germans,Q183,Germany,...,f750a83efdd7892cfd55d622c58c12c3e4eeee7b,2013-02-04 21:27:53+00:00,human,Zolo,4610,,1,0E-24,10000.0,1
6,5792391,5758,Bad Aibling,30,q5758$2A9787C5-83D9-43A1-B502-7C0B2FAE5A11,continent,Q218640,Europa,Q446913,Kungsholm,...,146e95be8f682d9d00ab357904fe156b00f327a8,2013-02-04 21:29:00+00:00,human,Jwdietrich2,6818,,1,0E-24,10000.0,1
7,5792399,5758,Bad Aibling,30,q5758$2A9787C5-83D9-43A1-B502-7C0B2FAE5A11,continent,Q446913,Kungsholm,Q689442,SS Europa,...,9e6619f819e885e4c02fb78537fec2ead3a8f7b4,2013-02-04 21:29:21+00:00,human,Jwdietrich2,6818,,1,0.000243055555555555560000,10000.0,1
8,5792402,5758,Bad Aibling,30,q5758$2A9787C5-83D9-43A1-B502-7C0B2FAE5A11,continent,Q689442,SS Europa,Q1129332,MS Europa,...,a33a5a0d7b27be1d66ecc0b1190bc6d19873eec0,2013-02-04 21:29:39+00:00,human,Jwdietrich2,6818,,1,0.000451388888888888890000,10000.0,5
9,5792408,5758,Bad Aibling,30,q5758$2A9787C5-83D9-43A1-B502-7C0B2FAE5A11,continent,Q1129332,MS Europa,Q5401,Eurasia,...,a7d1fff395b2ae0afe6f56885ca3eb700f32b788,2013-02-04 21:29:54+00:00,human,Jwdietrich2,6818,,1,0.000625000000000000000000,10000.0,1


In [ ]:
df_cluster_1 = df[df['cluster_id'] == 1]
print(df_cluster_1.size)
df_cluster_1[['property_id', 'property_label', 'old_value_label', 'new_value_label']]

# Cluster 1: value correction


7038775


,property_id,property_label,old_value_label,new_value_label
1,21,sex or gender,Masculine,masculinity
2,27,None,Indian Motorcycle,Indians
4,27,None,Indians,India
5,27,None,Germans,Germany
6,30,continent,Europa,Kungsholm
...,...,...,...,...
443875,31,None,human,robot
443877,159,headquarters location,Tulpenfeld,Bonn
443878,106,None,nurse,health professional
443879,31,None,award,commemorative medal


In [ ]:
df_cluster_2 = df[df['cluster_id'] == 2]
print(df_cluster_2.size)
df_cluster_2[['property_id', 'property_label', 'old_value_label', 'new_value_label']]

# Cluster 2 : link fix

1800


,property_id,property_label,old_value_label,new_value_label
133040,19,None,La Place,LaPlace
183013,276,None,land mass,landmass
258343,734,None,De Lange,DeLange
258695,118,None,LaLiga,La Liga
269934,463,member of,Super M,SuperM
...,...,...,...,...
347404,166,award received,Big Brother Awards,BigBrotherAwards
423309,1830,None,RTS 1,RTS1
427981,734,None,De Lillo,DeLillo
429767,180,depicts,sugarcane,sugar cane


In [44]:
df_cluster_3 = df[df['cluster_id'] == 4]
print(df_cluster_3.size)
df_cluster_3[['property_id', 'property_label', 'old_value_label', 'new_value_label']]


2109525


,property_id,property_label,old_value_label,new_value_label
52,71,None,Place de la Riponne,Platypezidae
61,47,None,People's Republic of China,Guangdong
74,47,None,Catholic Church in Luxembourg,Germany
99,46,None,Matata Ponyo Mapon Augustin,Denis Sassou-Nguesso
109,7,None,"Louis I, Duke of Orléans",None
...,...,...,...,...
443818,69,None,National Autonomous University of Mexico,North High School
443834,3018,None,Virunga National Park,Goma
443835,31,None,type of chemical entity,group of stereoisomers
443866,1465,None,None,Category:Deaths in Connecticut


In [20]:
import psycopg2
from psycopg2 import sql

query = """
    SELECT comment 
    FROM revision r join value_change vc on r.revision_id = vc.revision_id 
    WHERE vc.action = 'UPDATE'
        and NOT(typo or value_refinement or formatting or reverted_edit or reversion or value_unrefinement or link_fix or property_replacement or precision_change or sign_change)
    """

conn = psycopg2.connect(dbname='wikidata_change_classification', user='postgres', password='postgres', host='172.16.64.23', port='5432')

try:
    cur = conn.cursor()
    cur.execute(query)
    
    # Fetch all results
    results = cur.fetchall()
    
    columns = [desc[0] for desc in cur.description]
    
    # Create DataFrame
    df = pd.DataFrame(results, columns=columns)
    
    # Save to CSV
    df.to_csv('comments_typo.csv', index=False)
    print(f"Saved {len(df)} rows to comments_typo.csv")
    
    cur.close()
    
except Exception as e:
    print(f"Error: {e}")
    
finally:
    conn.close()

Saved 2290205 rows to comments_typo.csv


In [22]:
df_comments = pd.read_csv('comments_typo.csv')
df_comments

# cleanup
# fix

,comment
0,/* wbsetclaim-update:2||1 */ [[Property:P569]]...
1,/* wbsetclaim-update:2||1 */ [[Property:P242]]...
2,/* restore:0||1956250187|Alan ffm */ Unwarrant...
3,/* restore:0||1956250187|Alan ffm */ Unwarrant...
4,/* wbsetclaim-update:2||1 */ [[Property:P1831]...
...,...
2290200,/* wbsetclaim-update:2||1 */ [[Property:P373]]...
2290201,/* wbsetclaim-update:2||1 */ [[Property:P1971]...
2290202,/* wbsetdescription-set:1|en */ municipality i...
2290203,/* wbsetclaim-update:2||1 */ [[Property:P6]]: ...
